In [1]:
#@title Upload BoolNet Rules File
from google.colab import files
import os, textwrap, csv

# 1)  Upload or reuse file
if 'file_rules' in globals() and os.path.exists(file_rules):
    print(f"  Using previously uploaded file: {file_rules}")
else:
    print(textwrap.dedent("""
        Select your rules file (e.g., 'rules_boolnet_XXX.txt').
        You only need to do this ONCE; the path will be stored in the variable 'file_rules'.
    """))
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(uploaded.keys())[0]
    print(f"  File uploaded and stored in variable 'file_rules': {file_rules}")

# 2)  Read rules and compute N and <K>
genes   = []        # list of target genes
exprs   = []        # list of expressions (factors)

with open(file_rules, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        tgt = row['targets'].strip()
        fac = row['factors'].strip()
        if tgt:                       # skip empty lines
            genes.append(tgt)
            exprs.append(fac)

N = len(genes)

# → translate each expression to extract tokens
tr_table = str.maketrans({'(': ' ', ')': ' ', '&': ' ', '|': ' ', '!': ' '})
k_list = []
for expr in exprs:
    # remove operators and split into tokens
    tokens = expr.translate(tr_table).split()
    # regulators = genes found in the expression
    regs   = {tok for tok in tokens if tok in genes}
    k_list.append(len(regs))

K_avg = sum(k_list) / N if N else 0.0

print(f"\n  The network contains {N} nodes.")
print(f"  Average K (mean number of regulators per node): {K_avg:.2f}")



Select your rules file (e.g., 'rules_boolnet_XXX.txt').
You only need to do this ONCE; the path will be stored in the variable 'file_rules'.



Saving nsclc_9_nodes.txt to nsclc_9_nodes.txt
  File uploaded and stored in variable 'file_rules': nsclc_9_nodes.txt

  The network contains 9 nodes.
  Average K (mean number of regulators per node): 1.89


In [3]:
# @title Fitting and validation of rules using real‐time fixed‐point calculation
import itertools
import os
import re
import csv
import textwrap
import datetime
import pathlib
import shutil
from typing import Dict, List, Tuple, Set
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd

try:
    from google.colab import files   # noqa
    _IN_COLAB = True
except ModuleNotFoundError:
    _IN_COLAB = False

# tqdm for progress
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# networkx for signed cycles
try:
    import networkx as nx
except ModuleNotFoundError:
    raise ModuleNotFoundError("networkx is required. Install with: pip install networkx")

SHOW_PROGRESS = True

def prog_iter(iterable, desc=None, total=None, leave=False):
    if not SHOW_PROGRESS or not _HAS_TQDM:
        return iterable
    return tqdm(iterable, desc=desc, total=total, leave=leave)

# ====== Configurable parameters per run (will be set via input) ======
MAX_VARS_PER_RULE = 4           # 1..4
ALLOW_SELF_IN_RULES = True      # self-reference

CPU_COUNT = os.cpu_count() or 1
N_WORKERS = 1  # will be asked and overwritten

# 0-bis) File utilities (local Jupyter)
def choose_rules_file() -> str:
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        path = filedialog.askopenfilename(
            title="Select the BoolNet file (txt/csv)",
            filetypes=[("Text/CSV", "*.txt *.csv"), ("All files", "*.*")]
        )
        root.destroy()
        if path:
            p = os.path.abspath(os.path.expanduser(path))
            if os.path.exists(p):
                return p
    except Exception as e:
        print(f"(Could not use graphical dialog: {e})")

    while True:
        p = input("Enter the full path to the rules file (txt/csv): ").strip().strip('"')
        if p:
            p = os.path.abspath(os.path.expanduser(p))
            if os.path.exists(p):
                return p
        print("Invalid path. Please try again.")

# 1) Load BoolNet file (rules) + Output TAG
if 'file_rules' not in globals() or not os.path.exists(str(file_rules)):
    if _IN_COLAB:
        print(textwrap.dedent("""\
            Upload your BoolNet rules file (e.g. 'boolnet_rules_8_nodes.txt').
            It will be accessible as a global variable 'file_rules'.
        """))
        up = files.upload()
        if not up:
            raise RuntimeError("You must upload a rules file.")
        file_rules = list(up.keys())[0]
    else:
        print("Select the rules file from your computer (local Jupyter).")
        file_rules = choose_rules_file()

print(f"Rules file: {file_rules}")

def _slug(s: str) -> str:
    s2 = re.sub(r'[^A-Za-z0-9._-]+', '_', s).strip('_')
    return s2[:100] if s2 else "input"

_INPUT_BASENAME = pathlib.Path(str(file_rules)).stem
_TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
TAG = f"{_slug(_INPUT_BASENAME)}__{_TIMESTAMP}"
print(f"Output label (TAG): {TAG}")

OUT_ROOT = "Fitting"
OUT_DIR = os.path.join(OUT_ROOT, TAG)
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUT_DIR}")

def _out(name: str, ext: str) -> str:
    base = f"{name}__{TAG}{ext}"
    return os.path.join(OUT_DIR, base)

def load_rules_boolnet(fname: str) -> Dict[str, str]:
    rules: Dict[str, str] = {}
    with open(fname, encoding='utf-8') as fh:
        first = fh.readline().strip()
        fh.seek(0)

        if ":" in first and not first.lower().startswith(("targets", "target")):
            for line in fh:
                if ":" not in line:
                    continue
                node, expr = map(str.strip, line.split(":", 1))
                rules[node] = expr
        else:
            for row in csv.reader(fh, delimiter=','):
                if not row or len(row) < 2:
                    continue
                node, expr = row[0].strip(), row[1].strip()
                if node.lower().startswith(("targets", "target")):
                    continue
                rules[node] = expr

    return rules

original_rules = load_rules_boolnet(file_rules)
nodes = list(original_rules)
print("Nodes:", nodes)

# 1-bis) Interactive search space config
def _ask_bool(prompt: str, default: bool) -> bool:
    d = "y" if default else "n"
    txt = input(f"{prompt} [y/n] (default={d}): ").strip().lower()
    if not txt:
        return default
    return txt in {"y", "yes", "1", "true", "t"}

def _ask_int(prompt: str, default: int, lo: int, hi: int) -> int:
    txt = input(f"{prompt} ({lo}-{hi}) (default={default}): ").strip()
    if not txt:
        return default
    try:
        val = int(txt)
    except ValueError:
        print(f"Invalid value, using {default}.")
        return default
    if val < lo or val > hi:
        print(f"Out of range, using {default}.")
        return default
    return val

print(f"\n Available CPU cores: {CPU_COUNT}")
N_WORKERS = _ask_int("→ How many CPU cores to use?", min(4, CPU_COUNT), 1, CPU_COUNT)

print("\n Configure search space (you can leave defaults):")
MAX_VARS_PER_RULE = _ask_int("→ Max variables per rule", MAX_VARS_PER_RULE, 1, 4)
ALLOW_SELF_IN_RULES = _ask_bool("→ Allow self-reference in rules?", ALLOW_SELF_IN_RULES)

print("\n Selected config:")
print(f"   ▸ CPU cores used       = {N_WORKERS}/{CPU_COUNT}")
print(f"   ▸ MAX_VARS_PER_RULE    = {MAX_VARS_PER_RULE}")
print(f"   ▸ ALLOW_SELF_IN_RULES  = {ALLOW_SELF_IN_RULES}")

# 2) Network evaluation and attractor search
def _subst(expr: str, values: Dict[str, int], order: List[str]) -> str:
    for v in order:
        expr = re.sub(rf'\b{re.escape(v)}\b', str(values[v]), expr)
    return expr

def evaluate_network(state: Tuple[int, ...], rules: Dict[str, str]) -> Tuple[int, ...]:
    ctx = {n: state[i] for i, n in enumerate(nodes)}
    order = sorted(nodes, key=len, reverse=True)
    return tuple(
        1 if eval(
            _subst(rules[n], ctx, order)
            .replace('!', ' not ')
            .replace('&', ' and ')
            .replace('|', ' or ')
        ) else 0
        for n in nodes
    )

def _canonical_cycle(cycle: Tuple[Tuple[int, ...], ...]) -> Tuple[Tuple[int, ...], ...]:
    if len(cycle) == 1:
        return cycle
    rotations = [cycle[i:] + cycle[:i] for i in range(len(cycle))]
    return min(rotations)

def find_attractors(rules: Dict[str, str]) -> Set[Tuple[Tuple[int, ...], ...]]:
    seen_global: Set[Tuple[int, ...]] = set()
    attractors: Set[Tuple[Tuple[int, ...], ...]] = set()

    for start in itertools.product([0, 1], repeat=len(nodes)):
        if start in seen_global:
            continue

        path: List[Tuple[int, ...]] = []
        seen_local = {}
        s = start

        while s not in seen_local and s not in seen_global:
            seen_local[s] = len(path)
            path.append(s)
            s = evaluate_network(s, rules)

        if s in seen_local:
            cyc = tuple(path[seen_local[s]:])
            attractors.add(_canonical_cycle(cyc))

        seen_global.update(path)

    return attractors

# 3) Select attractors to preserve
orig_attractors = sorted(find_attractors(original_rules), key=lambda c: (len(c), c))
print(f"\n Found attractors: {len(orig_attractors)}\n")

for i, atr in enumerate(orig_attractors, 1):
    if len(atr) == 1:
        print(f" {i:>2}. Fixed point : {atr[0]}")
    else:
        print(f" {i:>2}. Cycle {len(atr):<2}: {list(atr)}")

sel = input("\n Attractors to keep (indices, comma separated): ").strip()
if not sel:
    raise RuntimeError("You must specify at least one index.")

indices = {int(s) for s in re.split(r'[\s,]+', sel) if s}
if min(indices) < 1 or max(indices) > len(orig_attractors):
    raise ValueError("Indices out of range.")

desired_attractors = {orig_attractors[i - 1] for i in indices}
print("\n Keeping", len(desired_attractors), "attractors.")

def format_attractors_block(attractors, title="Found attractors") -> List[str]:
    lines = [f"{title}: {len(attractors)}", ""]
    for i, atr in enumerate(attractors, 1):
        if len(atr) == 1:
            lines.append(f" {i:>2}. Fixed point : {atr[0]}")
        else:
            lines.append(f" {i:>2}. Cycle {len(atr):<2}: {list(atr)}")
    return lines

# 3-ter) Labeling and BASINS utilities (FP + cycles, stable columns)
def _is_fixed_point(attr):
    return isinstance(attr, tuple) and attr and isinstance(attr[0], int)

def _label_state(state_tuple):
    return "[" + ",".join(map(str, state_tuple)) + "]"

def _label_attractor(attr):
    if _is_fixed_point(attr):
        return f"FP {_label_state(attr)}"
    cyc = _canonical_cycle(attr)
    chain = "→".join(_label_state(s) for s in cyc)
    return f"Cycle L={len(cyc)}: {chain}"

def _state_to_int(s):
    bitstr = "".join(str(b) for b in s)
    return int(bitstr, 2)

def _attractor_sort_key(key):
    if _is_fixed_point(key):
        return (0, _state_to_int(key))
    cyc = _canonical_cycle(key)
    mins = min(_state_to_int(s) for s in cyc)
    return (1, len(cyc), mins)

def basins_by_attractor(rules_dict):
    state_to_attr = {}

    def next_state(s):
        return evaluate_network(s, rules_dict)

    for start in itertools.product([0, 1], repeat=len(nodes)):
        if start in state_to_attr:
            continue

        seen_index = {}
        s = tuple(start)
        while s not in seen_index and s not in state_to_attr:
            seen_index[s] = len(seen_index)
            s = next_state(s)

        if s in state_to_attr:
            attr_key = state_to_attr[s]
        else:
            cycle = []
            t = s
            while True:
                cycle.append(t)
                t = next_state(t)
                if t == s:
                    break
            attr_key = cycle[0] if len(cycle) == 1 else _canonical_cycle(tuple(cycle))

        for st in seen_index:
            state_to_attr[st] = attr_key

    counts = Counter(state_to_attr.values())
    sizes = dict(counts)
    labels = {k: _label_attractor(k) for k in sizes.keys()}
    return sizes, labels

# 3-cuater) Count number of components used in a rule
def count_rule_components(rule: str, node_names: List[str]) -> int:
    found = {n for n in node_names if re.search(rf'\b{re.escape(n)}\b', rule)}
    return len(found)

# 4) Count topological cycles as Positive vs Negative (René Thomas)
def count_cycles_thomas(rules_dict) -> Dict[str, int]:
    genes = list(rules_dict.keys())
    gene_set = set(genes)

    signs_map: Dict[Tuple[str, str], Set[int]] = {}
    for tgt, fac in rules_dict.items():
        for m in re.finditer(r'(!)?([A-Za-z_]\w*)', fac):
            neg, tok = m.groups()
            if tok in gene_set:
                s = -1 if neg else 1
                signs_map.setdefault((tok, tgt), set()).add(s)

    G = nx.DiGraph()
    G.add_nodes_from(genes)
    for (u, v) in signs_map.keys():
        G.add_edge(u, v)

    cycles = list(nx.simple_cycles(G))

    pos = 0
    neg = 0

    for cyc in cycles:
        edges = list(zip(cyc, cyc[1:] + [cyc[0]]))
        possible_parities = {0}  # 0=even(positive), 1=odd(negative)

        for (u, v) in edges:
            sset = signs_map.get((u, v), {1})
            new_parities = set()
            for p in possible_parities:
                for s in sset:
                    new_parities.add((p + (1 if s == -1 else 0)) % 2)
            possible_parities = new_parities

        if 0 in possible_parities:
            pos += 1
        if 1 in possible_parities:
            neg += 1

    return {"positive": pos, "negative": neg}

# 5) Canonical normalization of expressions
_TOKEN_RE = re.compile(r'\s*([A-Za-z_]\w*|!|&|\||\(|\))')

def _tokenize(expr: str) -> List[str]:
    pos, toks = 0, []
    while pos < len(expr):
        m = _TOKEN_RE.match(expr, pos)
        if not m:
            raise ValueError(f"Unexpected token at: {expr[pos:pos+15]!r}")
        toks.append(m.group(1))
        pos = m.end()
    return toks

def _parse_expr(toks: List[str]):
    i = 0

    def peek():
        return toks[i] if i < len(toks) else None

    def eat(x=None):
        nonlocal i
        if x is not None and toks[i] != x:
            raise ValueError(f"Expected {x}, got {toks[i]}")
        t = toks[i]
        i += 1
        return t

    def parse_not():
        if peek() == '!':
            eat('!')
            return ('not', parse_not())
        return parse_primary()

    def parse_primary():
        t = peek()
        if t == '(':
            eat('(')
            node = parse_or()
            eat(')')
            return node
        if t is None:
            raise ValueError("Incomplete expression")
        eat()
        return ('var', t)

    def parse_and():
        left = parse_not()
        parts = [left]
        while peek() == '&':
            eat('&')
            parts.append(parse_not())
        return ('and', parts) if len(parts) > 1 else parts[0]

    def parse_or():
        left = parse_and()
        parts = [left]
        while peek() == '|':
            eat('|')
            parts.append(parse_and())
        return ('or', parts) if len(parts) > 1 else parts[0]

    node = parse_or()
    if i != len(toks):
        raise ValueError("Remaining tokens at the end.")
    return node

def _needs_parentheses_in_parent(child, parent_op: str) -> bool:
    return _precedence(child) < _precedence((parent_op, []))

def _sort_key_in_parent(child, parent_op: str):
    needs_par = 0 if _needs_parentheses_in_parent(child, parent_op) else 1

    kind = child[0]
    is_simple_var = 1 if kind == 'var' else 0
    complexity = -len(_to_str(child))  # longer/more structured first

    return (needs_par, is_simple_var, complexity, _to_str(child))

def _canon(node):
    kind = node[0]

    if kind == 'var':
        return node

    if kind == 'not':
        return ('not', _canon(node[1]))

    if kind in ('and', 'or'):
        op = kind
        kids = [_canon(k) for k in node[1]]

        flat = []
        for k in kids:
            if isinstance(k, tuple) and k[0] == op:
                flat.extend(k[1])
            else:
                flat.append(k)

        seen = set()
        uniq = []
        for k in flat:
            s = _to_str(k)
            if s not in seen:
                seen.add(s)
                uniq.append(k)

        uniq.sort(key=lambda x: _sort_key_in_parent(x, op))
        return (op, uniq) if len(uniq) > 1 else uniq[0]

    raise ValueError(f"Unknown node type: {kind}")

def _precedence(n):
    return {'var': 4, 'not': 3, 'and': 2, 'or': 1}.get(n[0], 0)

def _to_str(n):
    k = n[0]

    if k == 'var':
        return n[1]

    if k == 'not':
        child = n[1]
        s = _to_str(child)
        return '!' + s if child[0] in ('var', 'not') else '!' + '(' + s + ')'

    if k in ('and', 'or'):
        sep = ' & ' if k == 'and' else ' | '
        parts = []
        for c in n[1]:
            cs = _to_str(c)
            if _precedence(c) < _precedence(n):
                cs = '(' + cs + ')'
            parts.append(cs)
        return sep.join(parts)

    raise ValueError("Invalid node")

def canonicalize_expr(expr: str, node_names: List[str]) -> str:
    toks = _tokenize(expr)
    ast = _parse_expr(toks)
    can = _canon(ast)
    return _to_str(can)

# 6) Canonical generation of candidate expressions (1..4 vars)
OPS = ('&', '|')

def signed_literals(vc: Tuple[str, ...]):
    for negs in itertools.product([False, True], repeat=len(vc)):
        yield [f'!{v}' if neg else v for v, neg in zip(vc, negs)]

def candidate_expressions_for_vars(vc: Tuple[str, ...]) -> List[str]:
    k = len(vc)
    if k < 1 or k > 4:
        raise ValueError("Supported only for 1..4 variables")

    out: Set[str] = set()

    for lits in signed_literals(vc):
        if k == 1:
            out.add(lits[0])

        elif k == 2:
            x1, x2 = lits
            for op1 in OPS:
                expr = f"{x1} {op1} {x2}"
                out.add(canonicalize_expr(expr, node_names=nodes))

        elif k == 3:
            # Only 3 ways to choose the grouped pair:
            # (0,1)-2, (0,2)-1, (1,2)-0
            pairings = [
                ((0, 1), 2),
                ((0, 2), 1),
                ((1, 2), 0),
            ]
            for (i, j), kidx in pairings:
                x1, x2, x3 = lits[i], lits[j], lits[kidx]
                for op1, op2 in itertools.product(OPS, repeat=2):
                    expr = f"({x1} {op1} {x2}) {op2} {x3}"
                    out.add(canonicalize_expr(expr, node_names=nodes))

        elif k == 4:
            # Shape 1: chain   ((a op b) op c) op d
            chain_forms = [
                ((0, 1), 2, 3),
                ((0, 2), 1, 3),
                ((0, 3), 1, 2),
                ((1, 2), 0, 3),
                ((1, 3), 0, 2),
                ((2, 3), 0, 1),
            ]

            for (i, j), kidx, lidx in chain_forms:
                x1, x2, x3, x4 = lits[i], lits[j], lits[kidx], lits[lidx]
                for op1, op2, op3 in itertools.product(OPS, repeat=3):
                    expr = f"(({x1} {op1} {x2}) {op2} {x3}) {op3} {x4}"
                    out.add(canonicalize_expr(expr, node_names=nodes))

            # Shape 2: balanced   (a op b) op (c op d)
            balanced_forms = [
                ((0, 1), (2, 3)),
                ((0, 2), (1, 3)),
                ((0, 3), (1, 2)),
            ]

            for (i, j), (kidx, lidx) in balanced_forms:
                x1, x2, x3, x4 = lits[i], lits[j], lits[kidx], lits[lidx]
                for op1, op2, op3 in itertools.product(OPS, repeat=3):
                    expr = f"({x1} {op1} {x2}) {op2} ({x3} {op3} {x4})"
                    out.add(canonicalize_expr(expr, node_names=nodes))

    return sorted(out)

def build_candidate_list(inputs_pool: List[str], max_vars: int) -> List[str]:
    inputs_pool = sorted(inputs_pool)
    seen = set()
    out = []

    for r in range(1, max_vars + 1):
        for combo in itertools.combinations(inputs_pool, r):
            for cand in candidate_expressions_for_vars(combo):
                if cand not in seen:
                    seen.add(cand)
                    out.append(cand)

    return out

# 7) Evaluate expression in a state
def eval_expr(expr: str, row, order: List[str]) -> int:
    expr_r = expr
    for col in order:
        expr_r = re.sub(rf'\b{re.escape(col)}\b', str(row[col]), expr_r)
    expr_r = expr_r.replace('!', ' not ').replace('&', ' and ').replace('|', ' or ')
    return int(eval(expr_r))

def _worker_check_candidate_monotarget(args):
    (cand, order_cols, df_states_rows, df_next_target_vals, original_rules_obj, target, desired_attractors_obj) = args

    try:
        res = [eval_expr(cand, row, order_cols) for row in df_states_rows]
        is_functional = all((r == exp) for r, exp in zip(res, df_next_target_vals))
    except Exception:
        is_functional = False

    is_exact = False
    if is_functional:
        tmp = original_rules_obj.copy()
        tmp[target] = cand
        try:
            is_exact = (find_attractors(tmp) == desired_attractors_obj)
        except Exception:
            is_exact = False

    return (cand, is_functional, is_exact)


# 8) Classic inference (1 node at a time) + BASINS
def monotarget_inference():
    log_lines: List[str] = []

    def log(msg: str):
        print(msg)
        log_lines.append(msg)

    log("\n=== Starting inference (1 node at a time) ===")

    # Add attractors found to the summary TXT
    for line in format_attractors_block(orig_attractors, "Found attractors"):
        log(line)

    log("")

    kept_attractors = [atr for atr in orig_attractors if atr in desired_attractors]
    for line in format_attractors_block(kept_attractors, "Selected attractors to keep"):
        log(line)

    log("")

    # Add original network positive/negative cycles to summary
    orig_cycle_counts = count_cycles_thomas(original_rules)
    log("  Original network topological cycles (René Thomas):")
    log(f"   Positive cycles : {orig_cycle_counts['positive']}")
    log(f"   Negative cycles : {orig_cycle_counts['negative']}")
    log("")

    log(f"Target attractors: {len(desired_attractors)}\n")

    # Basins summary accumulators
    sim_rows: List[Dict[str, object]] = []
    all_keys: Set[object] = set()

    # df_states (states) and df_next (next states)
    states, next_states = [], []
    for atr in desired_attractors:
        cyc = list(atr)
        for i, s in enumerate(cyc):
            states.append(s)
            next_states.append(cyc[(i + 1) % len(cyc)])

    df_states = pd.DataFrame(states, columns=nodes)
    df_next = pd.DataFrame(next_states, columns=nodes)

    order_cols = sorted(nodes, key=len, reverse=True)
    results = []

    df_states_rows = df_states.to_dict(orient="records")
    node_iter = prog_iter(nodes, desc=f"Nodes (cores={N_WORKERS})", total=len(nodes), leave=False)

    for target in node_iter:
        log(f"Processing node {target}…")

        original = original_rules[target]
        inputs = [n for n in nodes if n != target]
        inputs_pool = inputs + [target] if ALLOW_SELF_IN_RULES else inputs

        max_comb = min(MAX_VARS_PER_RULE, len(inputs_pool))
        candidate_list = build_candidate_list(inputs_pool, max_comb)

        total = 0
        functional = 0
        exact_matches = 0
        valid = []

        df_next_target_vals = list(df_next[target].astype(int).values)

        log(f"   Unique candidates to evaluate: {len(candidate_list)}")

        # ---------------- Parallel branch ----------------
        if N_WORKERS > 1 and len(candidate_list) > 0:
            tasks = [
                (
                    cand,
                    order_cols,
                    df_states_rows,
                    df_next_target_vals,
                    original_rules,
                    target,
                    desired_attractors
                )
                for cand in candidate_list
            ]

            with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
                futures = [ex.submit(_worker_check_candidate_monotarget, t) for t in tasks]

                for fut in prog_iter(
                    as_completed(futures),
                    desc=f"{target}: candidates (parallel)",
                    total=len(futures),
                    leave=False
                ):
                    cand, is_functional, is_exact = fut.result()
                    total += 1

                    results.append({
                        "Node": target,
                        "Rule": cand,
                        "Rules": count_rule_components(cand, nodes),
                        "Functional": is_functional,
                        "Exact": is_exact
                    })

                    if is_functional:
                        functional += 1
                        if is_exact:
                            exact_matches += 1
                            valid.append(cand)

        # ---------------- Sequential branch ----------------
        else:
            for cand in prog_iter(
                candidate_list,
                desc=f"{target}: candidates",
                total=len(candidate_list),
                leave=False
            ):
                total += 1

                try:
                    res = [eval_expr(cand, row, order_cols) for row in df_states_rows]
                    is_functional = all((r == exp) for r, exp in zip(res, df_next_target_vals))
                except Exception:
                    is_functional = False

                is_exact = False
                if is_functional:
                    functional += 1
                    tmp = original_rules.copy()
                    tmp[target] = cand
                    if find_attractors(tmp) == desired_attractors:
                        is_exact = True
                        exact_matches += 1
                        valid.append(cand)

                results.append({
                    "Node": target,
                    "Rule": cand,
                    "Rules": count_rule_components(cand, nodes),
                    "Functional": is_functional,
                    "Exact": is_exact
                })

        log(f"   Total combinations evaluated: {total}")
        log(f"   Functional                  : {functional}")
        log(f"   Exact                       : {exact_matches}")
        log(f"   Original rule               : {target} = {original}")

        if valid:
            log("Valid rules:")
            for v in sorted(set(valid)):
                log("      " + v)

            uniques = sorted(set(valid))

            for vrule_can in uniques:
                tmp_rules = original_rules.copy()
                tmp_rules[target] = vrule_can
                sizes, labels = basins_by_attractor(tmp_rules)
                cyc_counts = count_cycles_thomas(tmp_rules)

                sim_rows.append({
                    "Node": target,
                    "Original_rule": f"{target} = {original}",
                    "Found_rule": vrule_can,
                    "sizes": sizes,
                    "labels": labels,
                    "Positive_cycles": cyc_counts["positive"],
                    "Negative_cycles": cyc_counts["negative"],
                })
                all_keys.update(sizes.keys())
        else:
            log("No valid rules.")

        log("")

    # Save results (all tested rules)
    df_res = pd.DataFrame(results)
    csv_name = _out("All_rules", ".csv")
    txt_name = _out("Summary_inference", ".txt")
    df_res.to_csv(csv_name, index=False)

    with open(txt_name, "w", encoding="utf-8") as fh:
        fh.write("\n".join(log_lines))

    # Build basins summary table
    sim_csv = None
    if sim_rows:
        sorted_keys = sorted(all_keys, key=_attractor_sort_key)

        key_to_label = {}
        for row in sim_rows:
            for k, lbl in row["labels"].items():
                key_to_label.setdefault(k, lbl)

        attractor_cols = [key_to_label[k] for k in sorted_keys]

        extra_cols = [
            "Positive_cycles",
            "Negative_cycles",
        ]

        cols = ["Node", "Original_rule", "Found_rule"] + extra_cols + attractor_cols

        out_rows = []
        for r in sim_rows:
            d = {
                "Node": r["Node"],
                "Original_rule": r["Original_rule"],
                "Found_rule": r["Found_rule"],
            }

            for c in extra_cols:
                d[c] = r.get(c, 0)

            for k, colname in zip(sorted_keys, attractor_cols):
                d[colname] = r["sizes"].get(k, 0)

            out_rows.append(d)

        df_sim = pd.DataFrame(out_rows, columns=cols)
        sim_csv = _out("Summary_simulations_basins", ".csv")
        df_sim.to_csv(sim_csv, index=False)
        print(f"Basins summary CSV (fixed columns per attractor): {sim_csv}")
    else:
        print("No valid rules; no basins summary generated.")

    print(f"CSV: {csv_name}")
    print(f"TXT: {txt_name}")

    zip_base_name = f"Fitting__{TAG}"
    zip_path = shutil.make_archive(
        base_name=zip_base_name,
        format="zip",
        root_dir=OUT_ROOT,
        base_dir=TAG
    )
    print(f"ZIP created: {zip_path}")
    print("=== End monotarget inference ===\n")

    return csv_name, txt_name, zip_path

# 9) Execution (no automatic downloads)
csv_file, txt_file, zip_file = monotarget_inference()

Rules file: nsclc_9_nodes.txt
Output label (TAG): nsclc_9_nodes__20260720-163722
Output folder: Fitting/nsclc_9_nodes__20260720-163722
Nodes: ['miR_145', 'Sp1', 'MALAT1', 'BMI1', 'KLF4', 'p53', 'p53_A', 'p53_K', 'E2F1']

 Available CPU cores: 2
→ How many CPU cores to use? (1-2) (default=2): 2

 Configure search space (you can leave defaults):
→ Max variables per rule (1-4) (default=4): 3
→ Allow self-reference in rules? [y/n] (default=y): n

 Selected config:
   ▸ CPU cores used       = 2/2
   ▸ MAX_VARS_PER_RULE    = 3
   ▸ ALLOW_SELF_IN_RULES  = False

 Found attractors: 4

  1. Fixed point : (0, 1, 1, 1, 1, 0, 0, 0, 1)
  2. Fixed point : (1, 0, 0, 0, 0, 1, 0, 1, 0)
  3. Fixed point : (1, 0, 0, 0, 0, 1, 1, 0, 0)
  4. Cycle 2 : [(1, 0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 1, 1, 1, 0)]

 Attractors to keep (indices, comma separated): 1,2,3

 Keeping 3 attractors.

=== Starting inference (1 node at a time) ===
Found attractors: 4

  1. Fixed point : (0, 1, 1, 1, 1, 0, 0, 0, 1)
  2. Fi

Nodes (cores=2):   0%|          | 0/9 [00:00<?, ?it/s]

Processing node miR_145…
   Unique candidates to evaluate: 3824


miR_145: candidates (parallel):   0%|          | 0/3824 [00:00<?, ?it/s]

   Total combinations evaluated: 3824
   Functional                  : 857
   Exact                       : 7
   Original rule               : miR_145 = p53 & !MALAT1 & !BMI1
Valid rules:
      (!p53_A | !p53_K) & !BMI1
      (!p53_A | !p53_K) & !KLF4
      (!p53_A | !p53_K) & !MALAT1
      (p53_A | p53_K) & !BMI1
      (p53_A | p53_K) & !KLF4
      (p53_A | p53_K) & !MALAT1
      (p53_A | p53_K) & !Sp1

Processing node Sp1…
   Unique candidates to evaluate: 3824


...

   Total combinations evaluated: 3824
   Functional                  : 91
   Exact                       : 0
   Original rule               : p53_A = !p53_K & p53
No valid rules.

Processing node p53_K…
   Unique candidates to evaluate: 3824


p53_K: candidates (parallel):   0%|          | 0/3824 [00:00<?, ?it/s]

   Total combinations evaluated: 3824
   Functional                  : 91
   Exact                       : 0
   Original rule               : p53_K = !p53_A & p53
No valid rules.

Processing node E2F1…
   Unique candidates to evaluate: 3824


E2F1: candidates (parallel):   0%|          | 0/3824 [00:00<?, ?it/s]

   Total combinations evaluated: 3824
   Functional                  : 857
   Exact                       : 8
   Original rule               : E2F1 = MALAT1
Valid rules:
      !p53_A & !p53_K | !miR_145
      !p53_A & !p53_K | KLF4
      !p53_A & !p53_K | MALAT1
      !p53_A & !p53_K | Sp1
      p53_A & p53_K | !miR_145
      p53_A & p53_K | KLF4
      p53_A & p53_K | MALAT1
      p53_A & p53_K | Sp1

Basins summary CSV (fixed columns per attractor): Fitting/nsclc_9_nodes__20260720-163722/Summary_simulations_basins__nsclc_9_nodes__20260720-163722.csv
CSV: Fitting/nsclc_9_nodes__20260720-163722/All_rules__nsclc_9_nodes__20260720-163722.csv
TXT: Fitting/nsclc_9_nodes__20260720-163722/Summary_inference__nsclc_9_nodes__20260720-163722.txt
ZIP created: /content/Fitting__nsclc_9_nodes__20260720-163722.zip
=== End monotarget inference ===

